# Stage 3 Prompt Sweep Clean (Visibility v6 Family, Kaggle)

Leakage-free full-val prompt sweep on Qwen2.5-VL-3B-Instruct. All prompt variants use `_nocroppath` user templates, and the notebook fails fast if any selected prompt still exposes `crop_path`.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
JSONL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

BACKEND_MODE = "qwen_hf"
RUN_ID_PREFIX = "stage3_qwen_val_v2_sweep_v6_clean"

PROMPT_VERSIONS = [
    "qwen_vlm_labels_v1_prompt_v3_nocroppath",
    "qwen_vlm_labels_v1_prompt_v4_nocroppath",
    "qwen_vlm_labels_v1_prompt_v5a_visibility_gate_best_nocroppath",
    "qwen_vlm_labels_v1_prompt_v6a_notaglock_nocroppath",
    "qwen_vlm_labels_v1_prompt_v6b_positive_ambiguous_nocroppath",
    "qwen_vlm_labels_v1_prompt_v6c_balanced_nocroppath",
    "qwen_vlm_labels_v1_prompt_v6d_balanced_notaglock_nocroppath",
    "qwen_vlm_labels_v1_prompt_v6e_partial_geometry_nocroppath",
]

CONTROL_VERSION = "qwen_vlm_labels_v1_prompt_v5a_visibility_gate_best_nocroppath"

print("Prompt sweep size:", len(PROMPT_VERSIONS))
from IPython.display import display


In [ ]:
def sh(cmd: str, cwd: Path | None = None, check: bool = True):
    print(f"$ {cmd}")
    p = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return p


def require_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def metrics_row(prompt_version: str, run_id: str, run_dir: Path, eval_dir: Path) -> dict:
    metrics = read_json(eval_dir / "metrics.json")
    run_summary = read_json(run_dir / "run_summary.json")
    rates = metrics.get("rates", {})
    f1 = metrics.get("f1", {})

    row = {
        "prompt_version": prompt_version,
        "run_id": run_id,
        "parse_success": float(rates.get("parse_success_rate", 0.0)),
        "schema_valid": float(rates.get("schema_valid_rate", 0.0)),
        "coarse_acc": float(rates.get("coarse_class_accuracy", 0.0)),
        "coarse_macro_f1": float(f1.get("coarse_class_macro_f1", 0.0)),
        "visibility_acc": float(rates.get("visibility_accuracy", 0.0)),
        "visibility_macro_f1": float(f1.get("visibility_macro_f1", 0.0)),
        "needs_review_acc": float(rates.get("needs_review_accuracy", 0.0)),
        "tag_exact": float(rates.get("tag_exact_match_rate", 0.0)),
        "tag_mean_jaccard": float(rates.get("tag_mean_jaccard", 0.0)),
        "pred_ambiguous_rate": float(rates.get("pred_ambiguous_rate", 0.0)),
        "gt_ambiguous_rate": float(rates.get("gt_ambiguous_rate", 0.0)),
        "backend": run_summary.get("backend_mode_effective"),
        "model": run_summary.get("backend_settings_effective", {}).get("model"),
    }
    row["abs_ambiguous_gap"] = abs(row["pred_ambiguous_rate"] - row["gt_ambiguous_rate"])
    return row


def pass_rule(row: dict) -> bool:
    return (
        row["parse_success"] == 1.0
        and row["schema_valid"] == 1.0
        and row["visibility_macro_f1"] >= 0.45
        and 0.10 <= row["pred_ambiguous_rate"] <= 0.20
        and row["coarse_acc"] >= 0.9655
        and row["coarse_macro_f1"] >= 0.551
        and row["tag_mean_jaccard"] >= 0.34
    )


def soft_pass_rule(row: dict, control_row: dict) -> bool:
    return (
        row["visibility_macro_f1"] >= (control_row["visibility_macro_f1"] - 0.01)
        and row["abs_ambiguous_gap"] < control_row["abs_ambiguous_gap"]
        and row["coarse_acc"] >= 0.9655
    )


In [ ]:
# 1) Setup: clean clone + deps once
if REPO_DIR.exists():
    print("Removing existing repo to avoid stale prompt files:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
git_head = subprocess.check_output("git rev-parse --short HEAD", shell=True, cwd=str(REPO_DIR), text=True).strip()
print("git_head:", git_head)

sh("python -m pip install -q -U pip")
sh(f"python -m pip install -q -r {REPO_DIR / 'requirements.txt'}")
sh("python -m pip install -q -U transformers accelerate qwen-vl-utils")

sh("nvidia-smi", check=False)


In [ ]:
# 2) Resolve dataset path
VAL_JSONL = None
DATASET_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    p = root / JSONL_REL
    if p.exists():
        VAL_JSONL = p
        DATASET_ROOT = root
        break

if VAL_JSONL is None:
    found = list(Path('/kaggle/input').rglob('vlm_labels_v1_val_v2.annotated.jsonl'))
    if found:
        VAL_JSONL = found[0]
        DATASET_ROOT = VAL_JSONL.parents[2]

if VAL_JSONL is None:
    raise FileNotFoundError('Could not locate vlm_labels_v1_val_v2.annotated.jsonl under /kaggle/input')

print('VAL_JSONL:', VAL_JSONL)
print('DATASET_ROOT:', DATASET_ROOT)


In [ ]:
# 3) Hard preflight checks: registry wiring + clean content fingerprints per version
import yaml

cfg_path = REPO_DIR / "configs" / "pipeline" / "stage3_vlm_gt_baseline.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
versions = cfg["prompt"]["versions"]

expected_paths = {
    "qwen_vlm_labels_v1_prompt_v3_nocroppath": (
        "configs/pipeline/prompts/stage3_vlm_system_v3_visibility_tag_calibrated.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v3_visibility_tag_calibrated_nocroppath.txt",
    ),
    "qwen_vlm_labels_v1_prompt_v4_nocroppath": (
        "configs/pipeline/prompts/stage3_vlm_system_v4_visibility_recalibrated.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v4_visibility_recalibrated_nocroppath.txt",
    ),
    "qwen_vlm_labels_v1_prompt_v5a_visibility_gate_best_nocroppath": (
        "configs/pipeline/prompts/stage3_vlm_system_v5a_visibility_gate_best.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v5a_visibility_gate_best_nocroppath.txt",
    ),
    "qwen_vlm_labels_v1_prompt_v6a_notaglock_nocroppath": (
        "configs/pipeline/prompts/stage3_vlm_system_v6a_notaglock.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v6a_notaglock_nocroppath.txt",
    ),
    "qwen_vlm_labels_v1_prompt_v6b_positive_ambiguous_nocroppath": (
        "configs/pipeline/prompts/stage3_vlm_system_v6b_positive_ambiguous.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v6b_positive_ambiguous_nocroppath.txt",
    ),
    "qwen_vlm_labels_v1_prompt_v6c_balanced_nocroppath": (
        "configs/pipeline/prompts/stage3_vlm_system_v6c_balanced.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v6c_balanced_nocroppath.txt",
    ),
    "qwen_vlm_labels_v1_prompt_v6d_balanced_notaglock_nocroppath": (
        "configs/pipeline/prompts/stage3_vlm_system_v6d_balanced_notaglock.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v6d_balanced_notaglock_nocroppath.txt",
    ),
    "qwen_vlm_labels_v1_prompt_v6e_partial_geometry_nocroppath": (
        "configs/pipeline/prompts/stage3_vlm_system_v6e_partial_geometry.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v6e_partial_geometry_nocroppath.txt",
    ),
}

positive_sentence = 'If the visible evidence itself cannot be trusted due to blur, glare, washout, low contrast, heavy shadow, or unstable boundaries, prefer visibility="ambiguous" even when a conservative coarse_class guess is still possible.'
anti_sentence = 'Do not use visibility="ambiguous" only because defect evidence is weak, absent, or conservative while the visible region remains readable.'
taglock_line = 'If visibility="ambiguous", ensure the tags explain why.'

pattern_rules = {
    "qwen_vlm_labels_v1_prompt_v5a_visibility_gate_best_nocroppath": {
        "sys_must": [
            "Treat visibility as a property of visual interpretability and view completeness,",
            "Step 1: readability of the visible region",
        ],
        "usr_must": [taglock_line],
        "sys_must_not": [],
        "usr_must_not": ["crop_path"],
    },
    "qwen_vlm_labels_v1_prompt_v6a_notaglock_nocroppath": {
        "sys_must": [],
        "usr_must": [],
        "sys_must_not": [],
        "usr_must_not": [taglock_line, "crop_path"],
    },
    "qwen_vlm_labels_v1_prompt_v6b_positive_ambiguous_nocroppath": {
        "sys_must": [positive_sentence],
        "usr_must": [],
        "sys_must_not": [anti_sentence],
        "usr_must_not": ["crop_path"],
    },
    "qwen_vlm_labels_v1_prompt_v6c_balanced_nocroppath": {
        "sys_must": [positive_sentence, anti_sentence],
        "usr_must": [taglock_line],
        "sys_must_not": [],
        "usr_must_not": ["crop_path"],
    },
    "qwen_vlm_labels_v1_prompt_v6d_balanced_notaglock_nocroppath": {
        "sys_must": [positive_sentence, anti_sentence],
        "usr_must": [],
        "sys_must_not": [],
        "usr_must_not": [taglock_line, "crop_path"],
    },
    "qwen_vlm_labels_v1_prompt_v6e_partial_geometry_nocroppath": {
        "sys_must": [
            "clearly substantial part",
            "missing geometry or context",
            "Use visibility=\"partial\" only for readable but materially incomplete geometry/view; do not use partial for image-quality uncertainty.",
            "Readable fragment view with materially missing geometry is partial, not ambiguous.",
        ],
        "usr_must": [taglock_line],
        "sys_must_not": [],
        "usr_must_not": ["crop_path"],
    },
}

for pv in PROMPT_VERSIONS:
    if pv not in versions:
        raise RuntimeError(f"Prompt version missing in config: {pv}")

    expected_sys, expected_usr = expected_paths[pv]
    actual_sys = versions[pv]["system_path"]
    actual_usr = versions[pv]["user_path"]

    if actual_sys != expected_sys or actual_usr != expected_usr:
        raise RuntimeError(
            f"Registry mismatch for {pv}: expected ({expected_sys}, {expected_usr}), got ({actual_sys}, {actual_usr})"
        )

    sys_abs = REPO_DIR / actual_sys
    usr_abs = REPO_DIR / actual_usr
    require_path(sys_abs, f"system prompt for {pv}")
    require_path(usr_abs, f"user prompt for {pv}")

    sys_text = sys_abs.read_text(encoding="utf-8")
    usr_text = usr_abs.read_text(encoding="utf-8")
    if "crop_path" in usr_text:
        raise RuntimeError(f"Clean user prompt still exposes crop_path: {pv}")

    rules = pattern_rules.get(pv)
    if rules:
        for pattern in rules.get("sys_must", []):
            if pattern not in sys_text:
                raise RuntimeError(f"Pattern missing in system prompt {pv}: {pattern}")
        for pattern in rules.get("usr_must", []):
            if pattern not in usr_text:
                raise RuntimeError(f"Pattern missing in user prompt {pv}: {pattern}")
        for pattern in rules.get("sys_must_not", []):
            if pattern in sys_text:
                raise RuntimeError(f"Forbidden pattern in system prompt {pv}: {pattern}")
        for pattern in rules.get("usr_must_not", []):
            if pattern in usr_text:
                raise RuntimeError(f"Forbidden pattern in user prompt {pv}: {pattern}")

print("Clean prompt preflight checks passed for all sweep variants.")


In [ ]:
# 4) Run full sweep: run -> validate -> eval -> visuals for each prompt version
all_rows = []
run_dirs = {}

for pv in PROMPT_VERSIONS:
    run_id = f"{RUN_ID_PREFIX}_{pv}"
    print("\n" + "=" * 90)
    print("Running prompt version:", pv)
    print("run_id:", run_id)

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "run_stage3_vlm_baseline.py"),
            "--config", str(REPO_DIR / "configs" / "pipeline" / "stage3_vlm_gt_baseline.yaml"),
            "--backend-mode", BACKEND_MODE,
            "--prompt-version", pv,
            "--input-jsonl", f'"{VAL_JSONL}"',
            "--run-id", run_id,
            "--no-resume",
        ]),
        cwd=REPO_DIR,
    )

    run_dir = REPO_DIR / "outputs" / "stage3_vlm_baseline_runs" / run_id
    eval_dir = run_dir / "eval"
    pred_jsonl = run_dir / "predictions_vlm_labels_v1.jsonl"

    require_path(run_dir, f'run_dir for {pv}')
    require_path(pred_jsonl, f'predictions for {pv}')

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "validate_vlm_labels_v1.py"),
            "--input", f'"{pred_jsonl}"',
        ]),
        cwd=REPO_DIR,
    )

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "eval_stage3_vlm_baseline.py"),
            "--run-dir", f'"{run_dir}"',
            "--ground-truth-jsonl", f'"{VAL_JSONL}"',
        ]),
        cwd=REPO_DIR,
    )

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "visualize_stage3_eval_results.py"),
            "--eval-dir", f'"{eval_dir}"',
        ]),
        cwd=REPO_DIR,
    )

    row = metrics_row(prompt_version=pv, run_id=run_id, run_dir=run_dir, eval_dir=eval_dir)
    all_rows.append(row)
    run_dirs[pv] = run_dir

print('Sweep finished. Total runs:', len(all_rows))


In [ ]:
# 5) Aggregate + rank + PASS/SOFT_PASS/FAIL
if not all_rows:
    raise RuntimeError('No sweep rows collected.')

df = pd.DataFrame(all_rows)

control_matches = df[df['prompt_version'] == CONTROL_VERSION]
if control_matches.empty:
    raise RuntimeError(f'Control version not found in sweep rows: {CONTROL_VERSION}')
control_row = control_matches.iloc[0].to_dict()

verdicts = []
for _, r in df.iterrows():
    row = r.to_dict()
    if pass_rule(row):
        verdicts.append('PASS')
    elif soft_pass_rule(row, control_row):
        verdicts.append('SOFT_PASS')
    else:
        verdicts.append('FAIL')

df['verdict'] = verdicts

# Ranking rule:
# 1) highest visibility_macro_f1
# 2) smallest abs_ambiguous_gap
# 3) highest coarse_acc
# 4) highest tag_mean_jaccard
ranked = df.sort_values(
    by=['visibility_macro_f1', 'abs_ambiguous_gap', 'coarse_acc', 'tag_mean_jaccard'],
    ascending=[False, True, False, False],
).reset_index(drop=True)
ranked['rank'] = ranked.index + 1

best = ranked.iloc[0].to_dict()

AGG_DIR = REPO_DIR / 'outputs' / 'stage3_vlm_baseline_runs' / f'{RUN_ID_PREFIX}_aggregate'
AGG_DIR.mkdir(parents=True, exist_ok=True)

agg_csv = AGG_DIR / 'prompt_sweep_comparison.csv'
agg_md = AGG_DIR / 'prompt_sweep_comparison.md'
ranked.to_csv(agg_csv, index=False)

def to_markdown_table(df_in: pd.DataFrame) -> str:
    cols = list(df_in.columns)
    lines = []
    lines.append('| ' + ' | '.join(cols) + ' |')
    lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
    for _, rr in df_in.iterrows():
        vals = []
        for c in cols:
            v = rr[c]
            if isinstance(v, float):
                vals.append(f'{v:.6f}')
            else:
                vals.append(str(v))
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines)

md_lines = [
    '# Stage 3 Prompt Sweep Comparison',
    '',
    f'- control_version: `{CONTROL_VERSION}`',
    f'- best_prompt_version: `{best["prompt_version"]}`',
    f'- best_run_id: `{best["run_id"]}`',
    f'- best_verdict: `{best["verdict"]}`',
    '',
    to_markdown_table(ranked),
]
agg_md.write_text('\n'.join(md_lines), encoding='utf-8')

print('Aggregate CSV:', agg_csv)
print('Aggregate MD:', agg_md)
print('\n=== BEST CANDIDATE ===')
for k in ['prompt_version', 'run_id', 'verdict', 'visibility_macro_f1', 'abs_ambiguous_gap', 'coarse_acc', 'tag_mean_jaccard']:
    print(f'{k}: {best[k]}')

display(ranked)


In [ ]:
# 6) Package sweep deliverables
DELIVER_DIR = Path(f"/kaggle/working/stage3_deliverables_{RUN_ID_PREFIX}")
if DELIVER_DIR.exists():
    shutil.rmtree(DELIVER_DIR)
DELIVER_DIR.mkdir(parents=True, exist_ok=True)

for p in [AGG_DIR / 'prompt_sweep_comparison.csv', AGG_DIR / 'prompt_sweep_comparison.md']:
    if p.exists():
        dst = DELIVER_DIR / "aggregate" / p.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dst)

run_artifacts_rel = [
    "run_summary.json",
    "config_snapshot.json",
    "predictions_vlm_labels_v1.jsonl",
    "parsed_predictions.jsonl",
    "raw_responses.jsonl",
    "sample_results.jsonl",
    "failures.jsonl",
    "eval/metrics.json",
    "eval/confusion_coarse_class.csv",
    "eval/confusion_visibility.csv",
    "eval/review_table.csv",
    "eval/failures.jsonl",
    "eval/visuals/report.md",
    "eval/visuals/kpi_overview.png",
    "eval/visuals/confusion_coarse_class.png",
    "eval/visuals/confusion_visibility.png",
    "eval/visuals/visibility_errors_top.csv",
    "eval/visuals/visibility_errors_gallery.png",
]

for pv in PROMPT_VERSIONS:
    run_dir = run_dirs[pv]
    for rel in run_artifacts_rel:
        src_path = run_dir / rel
        if src_path.exists():
            dst = DELIVER_DIR / "runs" / pv / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_path, dst)

summary_md = DELIVER_DIR / "RESULT_SUMMARY.md"
summary_md.write_text(
    "\n".join([
        f"# Stage 3 Clean Prompt Sweep Result: {RUN_ID_PREFIX}",
        "",
        "- model: Qwen/Qwen2.5-VL-3B-Instruct",
        f"- backend: {BACKEND_MODE}",
        f"- control_version: {CONTROL_VERSION}",
        f"- prompt_count: {len(PROMPT_VERSIONS)}",
        f"- best_prompt_version: {best['prompt_version']}",
        f"- best_run_id: {best['run_id']}",
        f"- best_verdict: {best['verdict']}",
        "",
        f"Ground truth JSONL: {VAL_JSONL}",
        f"Repo commit: {git_head}",
    ]),
    encoding="utf-8",
)

ARCHIVE_BASE = Path(f"/kaggle/working/stage3_deliverables_{RUN_ID_PREFIX}")
ARCHIVE_PATH = shutil.make_archive(str(ARCHIVE_BASE), "gztar", root_dir=DELIVER_DIR)

print("DELIVER_DIR:", DELIVER_DIR)
print("ARCHIVE_PATH:", ARCHIVE_PATH)
print("\nFiles in deliverables:")
for p in sorted(DELIVER_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(DELIVER_DIR))


## Artifacts

Per-run outputs: `outputs/stage3_vlm_baseline_runs/<RUN_ID_PREFIX>...`
Aggregate files: `.../<RUN_ID_PREFIX>_aggregate/`
Packed archive: `/kaggle/working/stage3_deliverables_stage3_qwen_val_v2_sweep_v6_clean.tar.gz`
